# 07 Credit Policy

Aplicação do modelo em novas solicitações e definição de uma política de decisão de crédito.

## Abordagem

Este notebook aplica o modelo treinado à base de score e define a regra de política de crédito que vai orientar a aprovação das solicitações.


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))

import pandas as pd
from src.modeling import load_model
from src.policy import build_credit_policy, save_submission

score_features = pd.read_parquet(Path('..') / 'data' / 'processed' / 'score_feature_store.parquet')
model = load_model('final_model.pkl')

from src.feature_engineering import prepare_model_dataset
X_score, _, processed_score = prepare_model_dataset(score_features)
proba = model.predict_proba(X_score)[:, 1]

output = score_features[['id_cliente', 'data_solicitacao', 'tipo_contrato', 'valor_credito', 'valor_parcela']].copy()
output['probability'] = proba
output = build_credit_policy(output, cutoff=0.1)
save_submission(output, filename='submissao_case.csv')
print('Submission saved with shape', output.shape)
display(output.head())

## Regra de Negócio de Crédito

A política proposta usa um corte conservador de 10% de probabilidade de inadimplência.
As faixas de risco são agrupadas em: Very Low, Low, Medium, High e Very High.
A aprovação ocorre quando o score está abaixo do corte definido.